# Project Cuối Khóa: Lời Giải Gợi Ý - Dự Đoán Tình Trạng Vỡ Nợ Tín Dụng

**Lưu ý:** Đây là một lời giải gợi ý. Có nhiều cách tiếp cận khác nhau để giải quyết bài toán này, và kết quả của bạn có thể khác biệt tùy thuộc vào các lựa chọn trong quá trình tiền xử lý, không gian tìm kiếm tham số, và các yếu tố ngẫu nhiên. Mục tiêu chính là thể hiện một quy trình làm việc logic và hiệu quả.

### Bước 1: Khám phá và Tiền xử lý Dữ liệu

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix, precision_score, recall_score
from sklearn.preprocessing import LabelEncoder
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

In [ ]:
# 1. Tải dữ liệu
try:
    df = pd.read_csv('home_credit_risk_sample.csv')
except FileNotFoundError:
    print("Lỗi: Không tìm thấy file 'home_credit_risk_sample.csv'.")
    df = pd.DataFrame()

if not df.empty:
    # 2. Khám phá dữ liệu
    print(f"Kích thước dữ liệu: {df.shape}")
    
    # Bỏ cột ID không cần thiết cho mô hình
    if 'SK_ID_CURR' in df.columns:
        df = df.drop('SK_ID_CURR', axis=1)
        print("Đã xóa cột 'SK_ID_CURR'.")

    print("\nThông tin dữ liệu:")
    # df.info() # .info() in notebook prints to stdout, which can be verbose. Let's check head instead.
    print(df.head())

    # Kiểm tra sự mất cân bằng
    target_counts = df['TARGET'].value_counts(normalize=True) * 100
    print(f"\nTỷ lệ mất cân bằng của biến mục tiêu:\n{target_counts}")

    plt.figure(figsize=(6, 4))
    sns.countplot(x='TARGET', data=df)
    plt.title('Phân phối của biến mục tiêu (TARGET)')
    plt.show()

**Nhận xét:** Dữ liệu rất mất cân bằng, với chỉ khoảng 8% mẫu thuộc lớp `1` (vỡ nợ). Điều này khẳng định sự cần thiết phải sử dụng các kỹ thuật xử lý dữ liệu mất cân bằng.

In [ ]:
if not df.empty:
    # 3. Xử lý dữ liệu bị thiếu
    # Một chiến lược đơn giản: điền giá trị số bằng median và hạng mục bằng mode
    for col in df.columns:
        if df[col].dtype == 'object':
            df[col].fillna(df[col].mode()[0], inplace=True)
        else:
            df[col].fillna(df[col].median(), inplace=True)

    # 4. Xử lý Đặc trưng hạng mục
    # Sử dụng LabelEncoder cho tất cả các cột 'object'
    categorical_cols = [col for col in df.columns if df[col].dtype == 'object']
    le = LabelEncoder()
    for col in categorical_cols:
        df[col] = le.fit_transform(df[col])

    print("\nDữ liệu sau khi xử lý missing values và encoding:")
    print(df.head())
    print(f"\nSố lượng đặc trưng hạng mục đã mã hóa: {len(categorical_cols)}")

### Bước 2: Xây dựng Mô hình Cơ sở (Baseline Model)

In [ ]:
if not df.empty:
    # 1. Chia dữ liệu
    X = df.drop('TARGET', axis=1)
    y = df['TARGET']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

    # 2. Huấn luyện mô hình cơ sở
    # Tính scale_pos_weight
    scale_pos_weight = y_train.value_counts()[0] / y_train.value_counts()[1]
    print(f"Scale Pos Weight: {scale_pos_weight:.2f}")

    baseline_model = lgb.LGBMClassifier(
        objective='binary',
        metric='auc',
        scale_pos_weight=scale_pos_weight,
        random_state=42
    )

    baseline_model.fit(X_train, y_train)

    # 3. Đánh giá mô hình cơ sở
    y_pred_baseline = baseline_model.predict(X_test)
    y_pred_proba_baseline = baseline_model.predict_proba(X_test)[:, 1]

    print("\n--- Kết quả Mô hình Cơ sở ---")
    print(f"ROC AUC Score: {roc_auc_score(y_test, y_pred_proba_baseline):.4f}")
    print(f"F1 Score: {f1_score(y_test, y_pred_baseline):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred_baseline):.4f}")
    print(f"Recall: {recall_score(y_test, y_pred_baseline):.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred_baseline))

### Bước 3: Tinh chỉnh Siêu tham số (Hyperparameter Tuning)

In [ ]:
if not df.empty:
    # 1. Định nghĩa không gian tìm kiếm
    param_dist = {
        'n_estimators': [int(x) for x in np.linspace(start=100, stop=1000, num=10)],
        'learning_rate': [0.01, 0.05, 0.1],
        'num_leaves': [int(x) for x in np.linspace(start=20, stop=60, num=5)],
        'max_depth': [3, 5, 7, 9],
        'reg_alpha': [0.1, 0.5, 1],
        'reg_lambda': [0.1, 0.5, 1],
        'colsample_bytree': [0.6, 0.8, 1.0],
        'subsample': [0.6, 0.8, 1.0]
    }

    # 2. Chọn phương pháp: RandomizedSearchCV
    # Khởi tạo mô hình với các tham số cố định
    lgbm = lgb.LGBMClassifier(
        objective='binary',
        metric='auc',
        scale_pos_weight=scale_pos_weight,
        random_state=42
    )

    # 3. Thực hiện tìm kiếm
    random_search = RandomizedSearchCV(
        estimator=lgbm,
        param_distributions=param_dist,
        n_iter=20, # Giảm để chạy nhanh hơn, tăng để tìm kỹ hơn
        scoring='roc_auc',
        cv=3,
        n_jobs=-1,
        verbose=2,
        random_state=42
    )

    print("Bắt đầu tìm kiếm siêu tham số...")
    random_search.fit(X_train, y_train)

    # 4. Lấy kết quả
    print("\n--- Kết quả Tinh chỉnh Tham số ---")
    print(f"Bộ tham số tốt nhất: {random_search.best_params_}")
    print(f"Điểm AUC tốt nhất trên CV: {random_search.best_score_:.4f}")

### Bước 4: Huấn luyện và Đánh giá Mô hình Cuối cùng

In [ ]:
if not df.empty:
    # 1. Huấn luyện mô hình cuối cùng với bộ tham số tốt nhất
    final_model = lgb.LGBMClassifier(
        objective='binary',
        metric='auc',
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        **random_search.best_params_ # Sử dụng các tham số tốt nhất
    )

    final_model.fit(X_train, y_train)

    # 2. Đánh giá trên tập test
    y_pred_final = final_model.predict(X_test)
    y_pred_proba_final = final_model.predict_proba(X_test)[:, 1]

    print("\n--- Kết quả Mô hình Cuối cùng (sau khi tuning) ---")
    print(f"ROC AUC Score: {roc_auc_score(y_test, y_pred_proba_final):.4f}")
    print(f"F1 Score: {f1_score(y_test, y_pred_final):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred_final):.4f}")
    print(f"Recall: {recall_score(y_test, y_pred_final):.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred_final))

**So sánh:** So với mô hình cơ sở, mô hình cuối cùng thường sẽ có điểm ROC AUC cao hơn một chút, cho thấy khả năng phân loại tổng thể tốt hơn. Các chỉ số khác như F1, Precision, Recall cũng có thể thay đổi tùy thuộc vào bộ tham số tìm được.

### Bước 5: Diễn giải Mô hình

In [ ]:
if not df.empty:
    # 1. Feature Importance
    lgb.plot_importance(
        final_model, 
        importance_type='gain', 
        max_num_features=20, 
        figsize=(10, 12), 
        title='Feature Importance (Gain)'
    )
    plt.show()

    print("Nhận xét: Các đặc trưng như EXT_SOURCE_2, EXT_SOURCE_3, DAYS_BIRTH, và DAYS_EMPLOYED thường là những yếu tố quan trọng nhất. Điều này cho thấy điểm tín dụng từ các nguồn bên ngoài và thông tin cơ bản về tuổi tác, kinh nghiệm làm việc có ảnh hưởng lớn đến khả năng vỡ nợ.")

In [ ]:
if not df.empty:
    # 2. Phân tích với SHAP
    import shap

    # Tạo explainer và tính giá trị SHAP
    explainer = shap.TreeExplainer(final_model)
    shap_values = explainer.shap_values(X_test)

    # Vẽ summary plot
    # Chúng ta quan tâm đến lớp 1 (vỡ nợ)
    print("\n--- SHAP Summary Plot ---")
    shap.summary_plot(shap_values[1], X_test, max_display=20)

**Phân tích SHAP Summary Plot:**

- **`EXT_SOURCE_2` và `EXT_SOURCE_3`:** Rất rõ ràng, giá trị của các đặc trưng này càng thấp (màu xanh), SHAP value càng cao (dương), đẩy dự đoán về phía vỡ nợ. Ngược lại, điểm số cao từ các nguồn này (màu đỏ) giúp giảm rủi ro.
- **`DAYS_BIRTH`:** Đặc trưng này có giá trị âm (ngày sinh được đếm ngược từ ngày hiện tại). Giá trị cao hơn (màu đỏ, tức là người trẻ tuổi hơn) có SHAP value dương, cho thấy người trẻ có rủi ro vỡ nợ cao hơn. Người lớn tuổi hơn (giá trị thấp, màu xanh) có rủi ro thấp hơn.
- **`DAYS_EMPLOYED`:** Tương tự `DAYS_BIRTH`, giá trị cao hơn (màu đỏ, tức là người mới đi làm) có xu hướng làm tăng rủi ro vỡ nợ.

Những phân tích này cung cấp các insight rất giá trị cho bộ phận kinh doanh, giúp họ hiểu rõ các yếu tố rủi ro chính và có thể điều chỉnh chính sách cho vay cho phù hợp.

## Kết luận Tổng thể

Qua project này, chúng ta đã xây dựng thành công một quy trình học máy hoàn chỉnh để dự đoán rủi ro vỡ nợ tín dụng. Chúng ta đã bắt đầu bằng việc xử lý một bộ dữ liệu mất cân bằng, xây dựng một mô hình cơ sở, sau đó cải thiện nó thông qua việc tinh chỉnh siêu tham số một cách có hệ thống. Cuối cùng, chúng ta không chỉ dừng lại ở việc đưa ra dự đoán mà còn sử dụng các kỹ thuật diễn giải mô hình tiên tiến để hiểu sâu hơn về các quyết định của nó, mang lại những thông tin hữu ích có thể áp dụng vào thực tế kinh doanh.